In [33]:
import os
from openai import OpenAI
from agents import Agent, function_tool, SQLiteSession, Agent, Runner, trace, ModelSettings
from dotenv import load_dotenv
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import smtplib
import asyncio
from email.message import EmailMessage

load_dotenv(override=True)
MODEL_NAME = 'gpt-5.4-nano'

In [34]:
USER_EMAIL = os.getenv("GOOGLE_USER")
USER_PASS = os.getenv("GOOGLE_PASS")
EMAIL_SERVER = os.getenv("GOOGLE_SMTP")

In [35]:
USER_EMAIL, USER_PASS, EMAIL_SERVER

('mousamjha8@gmail.com', 'ruznnjixoqciqhnp', 'smtp.gmail.com')

In [36]:
@function_tool
def send_email(subject: str, text_body: str, html_body: str) -> str:
    """
        Send out the email with the given subject and body to all the sales prospects

        Agrs: 
            subject: The subject of the e-mail.
            text-body: The body of email as plain text
            html_body: THE HTML body of the email
    """
    msg = EmailMessage()
    msg['From'] = USER_EMAIL
    msg['To'] = USER_EMAIL
    msg['Subject'] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')
    print("Sending email.....")
    with smtplib.SMTP(EMAIL_SERVER, 587) as server:
        server.starttls()
        server.login(USER_EMAIL, USER_PASS)
        server.send_message(msg)
    print("Email sent......")

In [37]:
# send_email("Testing push email", "Hope email gets delivered!!!!", "<html><body><strong>Fingers</strong> crossed....</body></html>")

In [38]:
intro = """
    You are a sales agent working for Netshield,
    A company that provides SaaS tool for early fault prediction in 4G, 5G Sites, this application is powered by AI.
    You write emails.
"""

instruction1 = intro + "Your email style is professional, serious with gravitas and credibility."
instruction2 = intro + "Your email style is witty, engaging and humorous."
instruction3 = intro + "Your email style is concise, to the point, in the style busy senior executives do."

In [39]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instruction1, model=MODEL_NAME)
sales_agent2 = Agent(name="Witty Sales Agent", instructions=instruction2, model=MODEL_NAME)
sales_agent3 = Agent(name="Senior Sales Agent", instructions=instruction3, model=MODEL_NAME)

In [40]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email and for [Name] placeholder use Anonymus")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Early fault prediction for your 4G/5G sites (AI-powered)

Hi Anonymus,  

I’m reaching out from Netshield. We provide an AI-powered SaaS platform that predicts faults early across 4G and 5G sites, helping operators move from reactive maintenance to proactive prevention—reducing downtime and operational costs.

If you’re open to it, I’d like to ask one quick question:  
Are you currently using any predictive analytics or AI to detect network faults before they impact service?

Would a 15-minute call next week be worthwhile to see if Netshield fits your environment?

Best regards,  
[Your Name]  
Netshield | AI-driven fault prediction for 4G/5G  
[Phone] | [Email] | [Website]

In [41]:
result2 = Runner.run_streamed(sales_agent2, input="Write a witty email and in [Name] placeholder use Anonymus")
async for event in result2.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Quick question, [Name] — want fewer “surprise failures” in your 4G/5G sites? 😄

Hi [Name],

I hope you’re doing well! Quick one—how’s your luck been with “mysterious” faults lately? You know the ones: everything seems fine… until suddenly it isn’t. Like your network is auditioning for a horror movie.  

At Netshield, our AI-powered SaaS helps with **early fault prediction** for **4G/5G sites**, so you can catch issues before they turn into downtime, tickets, and that “how did this happen?!” meeting.

If you’re open to it, I can share a 2-minute overview and see if it fits your environment.  

Sound good?

Best,  
[Your Name]  
Netshield

In [42]:
result3 = Runner.run_streamed(sales_agent3, input="Write professional concise email")
async for event in result3.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Quick meeting to discuss AI early-fault prediction for 4G/5G

Hi [Name],  

I’m with Netshield. We provide an AI-powered SaaS platform that predicts early faults in 4G/5G sites to reduce downtime and improve operational efficiency.  

Are you open to a 15-minute call this week to see if it fits your network priorities?  

Best regards,  
[Your Name]  
Netshield  
[Phone] | [Email] | [Website]

In [43]:
message = "Write a cold sales email"

with trace("Prallel cold emails"):
    "TO DO: it runs agents in parallel and produce emails"
    result = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

output = [res.final_output for res in result]
for op in output:
    print(op + "\n\n")

Subject: Early fault prediction for your 4G/5G sites (AI-powered)

Hi [First Name],  

I’m [Your Name] from Netshield. We help operators and managed service teams predict faults on 4G/5G sites before they become incidents—using an AI-powered SaaS platform built for operational reliability.

Netshield helps you:  
- Detect early fault signals and reduce unplanned downtime  
- Prioritize sites most at risk with actionable insights  
- Improve maintenance planning and fault-handling efficiency  

If you’re open to it, I’d like to schedule 15 minutes to understand your current monitoring workflow and share how teams are using Netshield to catch faults earlier.

Would you be available [two time options]?

Best regards,  
[Your Name]  
Netshield | AI-powered early fault prediction  
[Phone] | [Email] | [Website]


Subject: Quick question about your 4G/5G alarms… (and your sleep schedule)  

Hi [Name],  

I’m [Your Name] from Netshield. We help operators predict faults early in 4G/5G sites us

In [44]:
decision = """
    You pick the best sales agent email for NettShield from the given options.
    Imagine you're a customer and pick the one you're most likely to respond to.
    Do, not add an explanation, reply with the selected email only.
""" 
sales_picker = Agent(name="Sales Email Picker", instructions=decision, model=MODEL_NAME)

In [45]:
message = "Write a professional NetShield email"
with trace("Sales email writer"):
    res = await asyncio.gather(
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message)
    )
    outputs = [result.final_output for result in res]
    emails = "NetShield sales email. \n\n + Email: \n\n" +",".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"The bes one that is selected is :\n\n {best}")

The bes one that is selected is :

 RunResult:
- Last agent: Agent(name="Sales Email Picker", ...)
- Final output (str):
    Subject: Early fault prediction for your 4G/5G sites
    
    Hi [Name],  
    
    NetShield helps operators predict equipment faults early across 4G/5G sites using AI—so teams can address issues before they impact availability and performance.  
    
    If you’re open to it, I’d like to schedule a quick 15-minute call to understand your current monitoring approach and share how NetShield could fit.  
    
    Are you available [Day/Time Option 1] or [Day/Time Option 2]?  
    
    Best regards,  
    [Your Name]  
    [Title], NetShield  
    [Phone] | [Email] | [Website]
- 1 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [46]:
send_email

FunctionTool(name='send_email', description='Send out the email with the given subject and body to all the sales prospects', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'text_body': {'title': 'Text Body', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'text_body', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11736abe0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [47]:
decision = """ 
    You pick the best NetSheild sales email from the given options.
    Imagine you're a customer, pick the best email and the most likely one that you would like to respond to.
    Then use the tool
"""

require_tools = ModelSettings(tool_choice="required", temperature=0.2)

In [48]:
sales_sender = Agent(name="Sales Email Sender", model=MODEL_NAME, model_settings=require_tools, instructions=decision, tools=[send_email])

In [49]:
message = "Write sales email for NetShield application"

with trace("Sales selection workflow with email sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message)
    )
    outputs = [result.final_output for result in results]
    emails = "NetShield Sales Emails:\n\n" + "\n\nEmail:\n\n" .join(outputs)
    response = await Runner.run(sales_sender, emails)
    print(f"Final result: \n {best.final_output}")

Sending email.....
Email sent......
Final result: 
 Subject: Early fault prediction for your 4G/5G sites

Hi [Name],  

NetShield helps operators predict equipment faults early across 4G/5G sites using AI—so teams can address issues before they impact availability and performance.  

If you’re open to it, I’d like to schedule a quick 15-minute call to understand your current monitoring approach and share how NetShield could fit.  

Are you available [Day/Time Option 1] or [Day/Time Option 2]?  

Best regards,  
[Your Name]  
[Title], NetShield  
[Phone] | [Email] | [Website]
